In [1]:
import os
import numpy as np
import pandas as pd
import gc

DATA_DIR_CANDIDATES = [
    "/kaggle/input/h-and-m-personalized-fashion-recommendations",
    "/kaggle/data",
    "/kaggle/input",
]


def _find_file(filename: str) -> str:
    for base in DATA_DIR_CANDIDATES:
        p = os.path.join(base, filename)
        if os.path.exists(p):
            return p
    for base in DATA_DIR_CANDIDATES:
        p = os.path.join(base, "h-and-m-personalized-fashion-recommendations", filename)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(
        f"Could not find {filename} in known Kaggle input locations."
    )


SAMPLE_PATH = _find_file("sample_submission.csv")
TRANS_PATH = _find_file("transactions_train.csv")

sample = pd.read_csv(SAMPLE_PATH)
sample = sample.sort_values("customer_id").reset_index(drop=True)

trans = pd.read_csv(
    TRANS_PATH,
    usecols=["t_dat", "customer_id", "article_id"],
    dtype={"t_dat": "string", "customer_id": "string", "article_id": "int32"},
)



In [2]:
# Change: add stronger but still simple fallback generators (recency + per-customer recency)
# to inject legitimate signal while keeping your blending core logic unchanged.


def build_global_top12(
    transactions: pd.DataFrame, sample_df: pd.DataFrame
) -> pd.DataFrame:
    top = transactions["article_id"].value_counts().head(12).index.astype(str).tolist()
    pred = " ".join(top)
    out = sample_df[["customer_id"]].copy()
    out["prediction"] = pred
    return out


def build_last_week_top12(
    transactions: pd.DataFrame, sample_df: pd.DataFrame
) -> pd.DataFrame:
    last_date = transactions["t_dat"].max()
    last_dt = pd.to_datetime(last_date)
    start_dt = (last_dt - pd.Timedelta(days=6)).strftime("%Y-%m-%d")
    last_week = transactions[transactions["t_dat"] >= start_dt]
    top = last_week["article_id"].value_counts().head(12).index.astype(str).tolist()
    if len(top) == 0:
        return build_global_top12(transactions, sample_df)
    pred = " ".join(top)
    out = sample_df[["customer_id"]].copy()
    out["prediction"] = pred
    return out


def build_recency_weighted_top12(
    transactions: pd.DataFrame,
    sample_df: pd.DataFrame,
    window_days: int = 30,
    decay: float = 0.08,
) -> pd.DataFrame:
    # Uses an exponential decay by age-in-days within a recent window; still a popularity list.
    t = transactions[["t_dat", "article_id"]].copy()
    t["t_dt"] = pd.to_datetime(t["t_dat"])
    last_dt = t["t_dt"].max()
    start_dt = last_dt - pd.Timedelta(days=window_days - 1)
    t = t[t["t_dt"] >= start_dt]
    if len(t) == 0:
        return build_global_top12(transactions, sample_df)
    age_days = (last_dt - t["t_dt"]).dt.days.astype("int16")
    w = np.exp(-decay * age_days.to_numpy())
    scores = pd.Series(w, index=t["article_id"].to_numpy()).groupby(level=0).sum()
    top = scores.sort_values(ascending=False).head(12).index.astype(str).tolist()
    pred = " ".join(top)
    out = sample_df[["customer_id"]].copy()
    out["prediction"] = pred
    return out


def build_customer_recent_top12(
    transactions: pd.DataFrame,
    sample_df: pd.DataFrame,
    global_fallback: list[str],
    max_items: int = 12,
) -> pd.DataFrame:
    # Predict each customer's most recent unique purchases (strong baseline for MAP@12),
    # then fill remaining slots with a global list to always output 12 items.
    t = transactions[["t_dat", "customer_id", "article_id"]].copy()
    t["t_dt"] = pd.to_datetime(t["t_dat"])
    # Sort so most recent comes first per customer; keep duplicates then drop to unique by article_id.
    t = t.sort_values(["customer_id", "t_dt"], ascending=[True, False])
    # Take a manageable recent slice per customer to keep runtime under control.
    # (Keeps core logic; avoids a full groupby-apply over 31M rows.)
    t = t.groupby("customer_id", sort=False).head(60)

    # Build per-customer list of unique articles preserving recency.
    t["article_id"] = t["article_id"].astype("int32")
    t = t.drop_duplicates(subset=["customer_id", "article_id"], keep="first")
    t = t.groupby("customer_id")["article_id"].apply(list).reset_index()

    # Merge into sample; customers without history get empty list.
    out = sample_df[["customer_id"]].merge(t, on="customer_id", how="left")
    out["article_id"] = out["article_id"].apply(
        lambda x: x if isinstance(x, list) else []
    )

    def _make_pred(lst):
        # convert to string, cap, then fill from global fallback without duplicates
        seen = set()
        res = []
        for a in lst:
            s = str(int(a))
            if s not in seen:
                seen.add(s)
                res.append(s)
            if len(res) >= max_items:
                break
        if len(res) < max_items:
            for g in global_fallback:
                if g not in seen:
                    seen.add(g)
                    res.append(g)
                if len(res) >= max_items:
                    break
        return " ".join(res[:max_items])

    out["prediction"] = out["article_id"].apply(_make_pred)
    return out[["customer_id", "prediction"]]


def safe_read_submission(
    path: str, sample_df: pd.DataFrame, fallback_kind: str
) -> pd.DataFrame:
    """
    Try reading an existing submission file; if missing, generate a fallback.
    Ensures:
      - all customers from sample_submission
      - sorted by customer_id
      - columns: customer_id, prediction (string)
    """
    if path is not None and os.path.exists(path):
        df = pd.read_csv(path)
        if "customer_id" not in df.columns or "prediction" not in df.columns:
            raise ValueError(
                f"{path} must contain columns ['customer_id','prediction']"
            )
        df["prediction"] = df["prediction"].astype(str)
        df = df.merge(sample_df[["customer_id"]], on="customer_id", how="right")
        df["prediction"] = df["prediction"].fillna("")
        df = df.sort_values("customer_id").reset_index(drop=True)
        return df[["customer_id", "prediction"]]

    # Precompute global fallback list once per call site (cheap and deterministic).
    global_top = trans["article_id"].value_counts().head(12).index.astype(str).tolist()

    if fallback_kind == "global":
        df = build_global_top12(trans, sample_df)
    elif fallback_kind == "last_week":
        df = build_last_week_top12(trans, sample_df)
    elif fallback_kind == "recency_weighted":
        df = build_recency_weighted_top12(trans, sample_df, window_days=30, decay=0.08)
    elif fallback_kind == "customer_recent":
        df = build_customer_recent_top12(
            trans, sample_df, global_fallback=global_top, max_items=12
        )
    else:
        df = build_global_top12(trans, sample_df)

    df["prediction"] = df["prediction"].astype(str)
    df = df.sort_values("customer_id").reset_index(drop=True)
    return df




In [3]:
# Change: make the four blended inputs meaningfully different but still in the same "submission blending" logic:
# - customer_recent (personalized)
# - last_week popularity
# - recency_weighted popularity
# - global popularity
sub0 = (
    safe_read_submission(path=None, sample_df=sample, fallback_kind="customer_recent")
    .sort_values("customer_id")
    .reset_index(drop=True)
)
sub1 = (
    safe_read_submission(path=None, sample_df=sample, fallback_kind="last_week")
    .sort_values("customer_id")
    .reset_index(drop=True)
)
sub2 = (
    safe_read_submission(path=None, sample_df=sample, fallback_kind="recency_weighted")
    .sort_values("customer_id")
    .reset_index(drop=True)
)
sub3 = (
    safe_read_submission(path=None, sample_df=sample, fallback_kind="global")
    .sort_values("customer_id")
    .reset_index(drop=True)
)



In [4]:
sub0.columns = ["customer_id", "prediction0"]
sub0["prediction1"] = sub1["prediction"]
sub0["prediction2"] = sub2["prediction"]
sub0["prediction3"] = sub3["prediction"].astype(str)



In [5]:
del sub1, sub2, sub3
gc.collect()
sub0.head()




,customer_id,prediction0,prediction1,prediction2,prediction3
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,568601043 841260003 887593002 890498002 795440...,909370001 865799006 918522001 448509014 751471...,751471001 915526001 918292001 706016001 909370...,706016001 706016002 372860001 610776002 759871...
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,826211002 599580055 811835004 723529001 559630...,909370001 865799006 918522001 448509014 751471...,751471001 915526001 918292001 706016001 909370...,706016001 706016002 372860001 610776002 759871...
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,858883002 851400006 750424014 870304002 852643...,909370001 865799006 918522001 448509014 751471...,751471001 915526001 918292001 706016001 909370...,706016001 706016002 372860001 610776002 759871...
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,742079001 732413001 706016001 706016002 372860...,909370001 865799006 918522001 448509014 751471...,751471001 915526001 918292001 706016001 909370...,706016001 706016002 372860001 610776002 759871...
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,896152002 730683050 927530004 791587015 589440...,909370001 865799006 918522001 448509014 751471...,751471001 915526001 918292001 706016001 909370...,706016001 706016002 372860001 610776002 759871...


In [6]:
def cust_blend(dt, W=[1, 1, 1, 1]):
    REC = []
    REC.append(str(dt["prediction0"]).split())
    REC.append(str(dt["prediction1"]).split())
    REC.append(str(dt["prediction2"]).split())
    REC.append(str(dt["prediction3"]).split())
    res = {}
    for M in range(len(REC)):
        for n, v in enumerate(REC[M]):
            if v in res:
                res[v] += W[M] / (n + 1)
            else:
                res[v] = W[M] / (n + 1)
    res = list(dict(sorted(res.items(), key=lambda item: -item[1])).keys())
    return " ".join(res[:12])


# Change: upweight the personalized generator slightly to move score upward toward target.
sub0["prediction"] = sub0.apply(cust_blend, W=[0.55, 0.20, 0.15, 0.10], axis=1)
sub0.head()



,customer_id,prediction0,prediction1,prediction2,prediction3,prediction
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,568601043 841260003 887593002 890498002 795440...,909370001 865799006 918522001 448509014 751471...,751471001 915526001 918292001 706016001 909370...,706016001 706016002 372860001 610776002 759871...,568601043 841260003 909370001 751471001 887593...
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,826211002 599580055 811835004 723529001 559630...,909370001 865799006 918522001 448509014 751471...,751471001 915526001 918292001 706016001 909370...,706016001 706016002 372860001 610776002 759871...,826211002 599580055 909370001 751471001 811835...
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,858883002 851400006 750424014 870304002 852643...,909370001 865799006 918522001 448509014 751471...,751471001 915526001 918292001 706016001 909370...,706016001 706016002 372860001 610776002 759871...,858883002 851400006 909370001 751471001 750424...
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,742079001 732413001 706016001 706016002 372860...,909370001 865799006 918522001 448509014 751471...,751471001 915526001 918292001 706016001 909370...,706016001 706016002 372860001 610776002 759871...,742079001 706016001 732413001 909370001 751471...
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,896152002 730683050 927530004 791587015 589440...,909370001 865799006 918522001 448509014 751471...,751471001 915526001 918292001 706016001 909370...,706016001 706016002 372860001 610776002 759871...,896152002 730683050 909370001 751471001 927530...


In [7]:
del sub0["prediction0"]
del sub0["prediction1"]
del sub0["prediction2"]
del sub0["prediction3"]
gc.collect()



0

In [8]:
# Second blend block preserved; use a different ordering/weighting but same cust_blend logic.
sub00 = (
    safe_read_submission(path=None, sample_df=sample, fallback_kind="customer_recent")
    .sort_values("customer_id")
    .reset_index(drop=True)
)
sub1 = (
    safe_read_submission(path=None, sample_df=sample, fallback_kind="recency_weighted")
    .sort_values("customer_id")
    .reset_index(drop=True)
)
sub2 = (
    safe_read_submission(path=None, sample_df=sample, fallback_kind="last_week")
    .sort_values("customer_id")
    .reset_index(drop=True)
)
sub3 = (
    safe_read_submission(path=None, sample_df=sample, fallback_kind="global")
    .sort_values("customer_id")
    .reset_index(drop=True)
)



In [9]:
sub00.columns = ["customer_id", "prediction0"]
sub00["prediction1"] = sub1["prediction"]
sub00["prediction2"] = sub2["prediction"]
sub00["prediction3"] = sub3["prediction"].astype(str)

del sub1, sub2, sub3
gc.collect()
sub00.head()



,customer_id,prediction0,prediction1,prediction2,prediction3
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,568601043 841260003 887593002 890498002 795440...,751471001 915526001 918292001 706016001 909370...,909370001 865799006 918522001 448509014 751471...,706016001 706016002 372860001 610776002 759871...
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,826211002 599580055 811835004 723529001 559630...,751471001 915526001 918292001 706016001 909370...,909370001 865799006 918522001 448509014 751471...,706016001 706016002 372860001 610776002 759871...
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,858883002 851400006 750424014 870304002 852643...,751471001 915526001 918292001 706016001 909370...,909370001 865799006 918522001 448509014 751471...,706016001 706016002 372860001 610776002 759871...
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,742079001 732413001 706016001 706016002 372860...,751471001 915526001 918292001 706016001 909370...,909370001 865799006 918522001 448509014 751471...,706016001 706016002 372860001 610776002 759871...
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,896152002 730683050 927530004 791587015 589440...,751471001 915526001 918292001 706016001 909370...,909370001 865799006 918522001 448509014 751471...,706016001 706016002 372860001 610776002 759871...


In [10]:
sub00["prediction"] = sub00.apply(cust_blend, W=[0.55, 0.15, 0.20, 0.10], axis=1)
sub00.head()



,customer_id,prediction0,prediction1,prediction2,prediction3,prediction
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,568601043 841260003 887593002 890498002 795440...,751471001 915526001 918292001 706016001 909370...,909370001 865799006 918522001 448509014 751471...,706016001 706016002 372860001 610776002 759871...,568601043 841260003 909370001 751471001 887593...
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,826211002 599580055 811835004 723529001 559630...,751471001 915526001 918292001 706016001 909370...,909370001 865799006 918522001 448509014 751471...,706016001 706016002 372860001 610776002 759871...,826211002 599580055 909370001 751471001 811835...
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,858883002 851400006 750424014 870304002 852643...,751471001 915526001 918292001 706016001 909370...,909370001 865799006 918522001 448509014 751471...,706016001 706016002 372860001 610776002 759871...,858883002 851400006 909370001 751471001 750424...
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,742079001 732413001 706016001 706016002 372860...,751471001 915526001 918292001 706016001 909370...,909370001 865799006 918522001 448509014 751471...,706016001 706016002 372860001 610776002 759871...,742079001 706016001 732413001 909370001 751471...
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,896152002 730683050 927530004 791587015 589440...,751471001 915526001 918292001 706016001 909370...,909370001 865799006 918522001 448509014 751471...,706016001 706016002 372860001 610776002 759871...,896152002 730683050 909370001 751471001 927530...


In [11]:
del sub00["prediction0"]
del sub00["prediction1"]
del sub00["prediction2"]
del sub00["prediction3"]
gc.collect()



0

In [12]:
sub1 = (
    safe_read_submission(path=None, sample_df=sample, fallback_kind="global")
    .sort_values("customer_id")
    .reset_index(drop=True)
)
sub1["prediction"] = sub1["prediction"].astype(str)
sub0["prediction"] = sub0["prediction"].astype(str)
sub00["prediction"] = sub00["prediction"].astype(str)

sub1.columns = ["customer_id", "prediction0"]
sub1["prediction1"] = sub0["prediction"]
sub1["prediction2"] = sub00["prediction"]
del sub0, sub00
gc.collect()




0

In [13]:
def cust_blend(dt, W=[1, 1, 1]):
    REC = []
    REC.append(str(dt["prediction0"]).split())
    REC.append(str(dt["prediction1"]).split())
    REC.append(str(dt["prediction2"]).split())
    res = {}
    for M in range(len(REC)):
        for n, v in enumerate(REC[M]):
            if v in res:
                res[v] += W[M] / (n + 1)
            else:
                res[v] = W[M] / (n + 1)
    res = list(dict(sorted(res.items(), key=lambda item: -item[1])).keys())
    return " ".join(res[:12])


# Change: final blend keeps your 3-way structure; bias toward the two improved blends.
sub1["prediction"] = sub1.apply(cust_blend, W=[0.10, 0.60, 0.55], axis=1)

del sub1["prediction0"]
del sub1["prediction1"]
del sub1["prediction2"]



In [14]:
sub1 = sub1.merge(sample[["customer_id"]], on="customer_id", how="right")
sub1["prediction"] = sub1["prediction"].fillna("").astype(str)
sub1 = sub1.sort_values("customer_id").reset_index(drop=True)
sub1 = sub1[["customer_id", "prediction"]]

sub1.to_csv("submission.csv", index=False)
print(sub1.head())
print("Wrote submission.csv with shape:", sub1.shape)

                                         customer_id  \
0  00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...   
1  0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...   
2  000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...   
3  00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...   
4  00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...   

                                          prediction  
0  568601043 841260003 909370001 751471001 706016...  
1  826211002 599580055 909370001 751471001 706016...  
2  858883002 851400006 909370001 751471001 706016...  
3  742079001 706016001 732413001 909370001 706016...  
4  896152002 730683050 909370001 751471001 706016...  
Wrote submission.csv with shape: (1371980, 2)
